> 在线讲义：[第 3 章 · 要测多少次：功效、最小可检测效应和样本量](https://wikieden.github.io/math-and-eval-stats/lessons/03-power-sample-size.html) · 本书《数学与实验统计》

# 第 3 章 · 要测多少次：功效、最小可检测效应和样本量

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/03-power-sample-size.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 分位数 | $z_p$ | `z_of(0.025)`、`z_of(0.20)` | 1.960、0.842 |
| 第二步 功效 | Φ(δ/SE − z) | `power_at(0.70, 0.85, 30)` | 0.293（模拟 0.280） |
| 第三步 样本量 | 两组公式 | `n_per_group(0.70, 0.85)` | 118；功效 90% 时 158 |
| 第四步 MDE | (z+z)√(2p(1−p)/n) | `mde(0.7, 30)` | 0.331 |
| 第三步 Miller | 式 (9) | `miller_n(1/9, 0.03)` | 969 |

例子：真成功率 0.70 vs 0.85（编的）；Miller 2024 式 (9) 的 969 题。
里程碑 ME-M2 的 0.6 / 0.7 功效模拟表留给自己做，这里不算。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
from scipy.stats import norm

**函数 `z_of`**：标准正态的上 p 分位数 z_p（z_0.025 = 1.96，z_0.20 = 0.84）。

In [ ]:
def z_of(p: float) -> float:
    """标准正态的上 p 分位数 z_p（z_0.025 = 1.96，z_0.20 = 0.84）。"""
    return float(norm.ppf(1 - p))

**函数 `n_per_group`**：两组独立比较、每组要多少次：n = (z_α/2 + z_β)² · (p1(1−p1) + p2(1−p2)) / δ²，向上取整。

In [ ]:
def n_per_group(p1: float, p2: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """两组独立比较、每组要多少次：n = (z_α/2 + z_β)² · (p1(1−p1) + p2(1−p2)) / δ²，向上取整。"""
    delta = abs(p1 - p2)
    var = p1 * (1 - p1) + p2 * (1 - p2)
    return math.ceil((z_of(alpha / 2) + z_of(1 - power)) ** 2 * var / delta**2)

**函数 `power_at`**：每组 n 次时的功效（正态近似）：Φ(δ / SE − z_α/2)，只算"往对的方向"那一侧。

In [ ]:
def power_at(p1: float, p2: float, n: int, alpha: float = 0.05) -> float:
    """每组 n 次时的功效（正态近似）：Φ(δ / SE − z_α/2)，只算"往对的方向"那一侧。"""
    se = math.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / n)
    return float(norm.cdf(abs(p1 - p2) / se - z_of(alpha / 2)))

**函数 `mde`**：每组 n 次、两组成功率都在 p 附近时，能可靠检出的最小差值 δ = (z_α/2 + z_β)·√(2p(1−p)/n)。

In [ ]:
def mde(p: float, n: int, alpha: float = 0.05, power: float = 0.8) -> float:
    """每组 n 次、两组成功率都在 p 附近时，能可靠检出的最小差值 δ = (z_α/2 + z_β)·√(2p(1−p)/n)。"""
    return (z_of(alpha / 2) + z_of(1 - power)) * math.sqrt(2 * p * (1 - p) / n)

**函数 `miller_n`**：Miller 2024 式 (9)（σ² 项取 0）：n = (z_α/2 + z_β)² · ω² / δ²。

In [ ]:
def miller_n(omega2: float, delta: float, alpha: float = 0.05, power: float = 0.8) -> float:
    """Miller 2024 式 (9)（σ² 项取 0）：n = (z_α/2 + z_β)² · ω² / δ²。"""
    return (z_of(alpha / 2) + z_of(1 - power)) ** 2 * omega2 / delta**2

**函数 `simulate_power`**：模拟：重做 reps 次实验，双比例 z 检验 p < 0.05 的比例。

In [ ]:
def simulate_power(p1: float, p2: float, n: int, reps: int = 20000, seed: int = 0) -> float:
    """模拟：重做 reps 次实验，双比例 z 检验 p < 0.05 的比例。"""
    rng = np.random.default_rng(seed)
    k1, k2 = rng.binomial(n, p1, reps), rng.binomial(n, p2, reps)
    pbar = (k1 + k2) / (2 * n)
    se = np.sqrt(pbar * (1 - pbar) * 2 / n)
    z = np.divide(k2 / n - k1 / n, se, out=np.zeros(reps), where=se > 0)
    return float(np.mean(np.abs(z) > z_of(0.025)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 两类错误：α 管误报，β 管漏报")
print(f"  z_0.025 = {z_of(0.025):.3f}，z_0.20 = {z_of(0.20):.3f}，z_0.10 = {z_of(0.10):.3f}")

In [ ]:
print("\n第二步 每组 30 次，0.70 vs 0.85 的功效")
print(f"  功效 = {power_at(0.70, 0.85, 30):.3f}（公式），模拟 {simulate_power(0.70, 0.85, 30):.3f}")

In [ ]:
print("\n第三步 想要 80% 功效，每组要多少次")
for power in (0.8, 0.9):
    print(f"  功效 {power:.0%}：每组 {n_per_group(0.70, 0.85, power=power)} 次")
print(f"  只想检出 5 个点（0.70 vs 0.75）：每组 {n_per_group(0.70, 0.75)} 次")

In [ ]:
print("\n第四步 次数定了，能检出多小的差（MDE）")
for n in (30, 100, 300):
    print(f"  每组 {n:3d} 次、成功率约 0.7：MDE = {mde(0.7, n):.3f}")

In [ ]:
print("\n第五步 Miller 2024 的例子：ω² = 1/9、δ = 0.03")
print(f"  n = {miller_n(1 / 9, 0.03):.0f} 道独立题")

## 3. 代码里才看得到的

1. **`z_of(1 - power)` 是 z_β**：功效 0.8 时 β = 0.2，要的是上 0.2 分位数 0.842；写成 `z_of(power)` 会得到 −0.842，样本量就算小了。
2. **公式只看"对的那一侧"**：双侧检验在"反方向显著"上也有一点点概率，真实功效比公式略高；差值不太小时可以忽略。
3. **模拟和公式差 1 个多点**：n = 30 时正态近似本身有误差（第 1 章说过 Wald 在小样本下不准），模拟值 0.280 比公式 0.293 略低。

## 4. 自己改一改

1. 把 `power_at` 的 n 从 30 改成 60，看功效是不是正好翻倍（不是，为什么？）。
2. 用 `mde(0.5, 100)` 和 `mde(0.9, 100)` 比一比：成功率接近 50% 时为什么最难检出差别？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 误报 α 漏报 β，没检出不等于没差别；
> 功效查表：差值除 SE 再减一点九六；
> 次数等于 z 和平方乘方差，再除差值平方；
> 开测先算 MDE，期望比它小，测了也白测。